In [2]:
import numpy as np
from itertools import chain
import pandas as pd
import json

ages_table_path = 'ages_table.xlsx'
ages_table = pd.read_excel(ages_table_path)

**Selecting subjects test**

In [7]:
# Selecting subjects test

range_groups = np.array([
    [41, 46],
    [46, 52],
    [52, 56.2],
    [56.2, 61],
    [61, 66],
    [66, 73],
    [73, 79],
    [79, 85],
    [85, 93],
    [93, 102],
    [102, 123],
    [123, 149],
    [149, 176],
    [176, 220],
    [220, 247],
    [247, 297],
    [297, 347],
    [347, 404],
    [404, 508],
    [508, 612],
    [612, 664],
    [664, 716],
    [716, 1082],
    [1082, 1499],
    [1499, 1708],
    [1708, 2021],
    [2021, 2125],
    [2125, 2699],
    [2699, 3220],
    [3220, 3481],
    [3481, 3689],
    [3689, 3900]
])
np.random.seed(7)
test_percent = 0.25
# select 10% subjects of each group to test
subjects_test = {} # dict subjects ages by intervals
subjects_train = {} # dict subjects ages by intervals
for interval in range_groups:
    
    start, end = interval
    subjects = ages_table.loc[(ages_table['age-weeks'] >= start) & (ages_table['age-weeks'] < end), "code"].tolist()
    start = str(start)
    end = str(end)
    if len(subjects) <= 3:
        quantity = 1
        files_test = np.random.choice(subjects, size=quantity, replace=False)
        subjects_test[f"{start}-{end}"] = files_test
    else:
        quantity = int(test_percent * len(subjects))
        files_test = list(np.random.choice(subjects, size=quantity, replace=False))
        subjects_test[f"{start}-{end}"] = files_test

with open("subjects_test.json", "w", encoding="utf-8") as f:
    json.dump(subjects_test, f, indent=4, ensure_ascii=False)

TypeError: Object of type int64 is not JSON serializable

**Define train subjects**

**Normalization parameters by age**

Note: let $x$ be age (years). For RR mean, RR standard deviation, and pNN50 use the formulas below.

RR mean:

$$
\text{RR\_mean} = 505 \cdot x^{0.122}
$$

For RR standard deviation and pNN50 use the age-dependent formulas:

If $x \le 12$:

$$
\text{RR\_std} = 80 \cdot x^{0.26} \\,
\text{pNN50} = 0.037 \cdot x^{0.78}
$$

If $x > 12$:

$$
\text{RR\_std} = 290 \cdot x^{-0.2} \\,
\text{pNN50} = 5 \cdot x^{-1.1}
$$

Python implementation:

```python
# x = age (years)
RR_mean = 505 * x**0.122
if x <= 12:
    RR_std = 80 * x**0.26
    pNN50 = 0.037 * x**0.78
else:
    RR_std = 290 * x**(-0.2)
    pNN50 = 5 * x**(-1.1)
```

In [5]:
import json
import numpy as np

# Selecting subjects test

range_groups = [
    (41, 46),
    (46, 52),
    (52, 56.2),
    (56.2, 61),
    (61, 66),
    (66, 73),
    (73, 79),
    (79, 85),
    (85, 93),
    (93, 102),
    (102, 123),
    (123, 149),
    (149, 176),
    (176, 220),
    (220, 247),
    (247, 297),
    (297, 347),
    (347, 404),
    (404, 508),
    (508, 612),
    (612, 664),
    (664, 716),
    (716, 1082),
    (1082, 1499),
    (1499, 1708),
    (1708, 2021),
    (2021, 2125),
    (2125, 2699),
    (2699, 3220),
    (3220, 3481),
    (3481, 3689),
    (3689, 3900)
]

np.random.seed(7)
test_percent = 0.3

subjects_test = {}
subjects_train = {}

for start, end in range_groups:
    subjects = ages_table.loc[
        (ages_table["age-weeks"] >= start) & (ages_table["age-weeks"] < end),
        "code"
    ].tolist()

    key = f"{start:g}-{end:g}"

    if len(subjects) <= 3:
        quantity = 1
    else:
        quantity = max(1, int(test_percent * len(subjects)))

    files_test = np.random.choice(subjects, size=quantity, replace=False).tolist()
    subjects_test[key] = [str(x) for x in files_test]

with open("subjects_test.json", "w", encoding="utf-8") as f:
    json.dump(subjects_test, f, indent=4, ensure_ascii=False)